### untuk cleaning data detik raw ama detik yg dh di label manual (masih tab separated)

In [1]:
import re
import pandas as pd

DETIK_DATA_PATH = '../../data/data_berita/cleaning/detik'

In [2]:
import pandas as pd

df_raw = pd.read_csv(
    f"{DETIK_DATA_PATH}/detik_labeled_manual_raw.tsv",
    sep="\t",
    encoding="utf-8",
    engine="python"
)

df_raw.head()


,date,title,content,text,label,row_id,label_suggest,label_suggest_v2,Unnamed: 8,Unnamed: 9
0,2023-12-20,mendagri copot pj bupati kampar karena tak net...,menteri dalam negeri mendagri tito karnavian m...,mendagri copot pj bupati kampar karena tak net...,negatif,7302,0,Negatif,neg,332.0
1,2025-06-20,waka mpr nilai keputusan prabowo absen di g7 j...,"wakil ketua mpr dari fraksi pan, eddy soeparno...",waka mpr nilai keputusan prabowo absen di g7 j...,netral,1543,-1,Positif,pos,363.0
2,2024-02-14,"setelah nyoblos, ahy bakal merapat ke istora s...",tim kampanye nasional tkn prabowo-gibran bakal...,"setelah nyoblos, ahy bakal merapat ke istora s...",netral,6055,-1,Positif,net,305.0
3,2024-06-05,obrolan jk dan baradar jika diizinkan saya ban...,wakil perdana menteri 1 afghanistan mullah abd...,obrolan jk dan baradar jika diizinkan saya ban...,netral,4993,0,Positif,NaN,NaN
4,2023-10-31,golkar masih lobi khofifah gabung tkn prabowo-...,sekjen partai golkar lodewijk f paulus berbica...,golkar masih lobi khofifah gabung tkn prabowo-...,netral,8269,0,Positif,NaN,NaN


In [13]:
display(df_raw.columns)
display(df_raw.shape)

Index(['date', 'title', 'content', 'text', 'label', 'row_id', 'label_suggest',
       'label_suggest_v2', 'Unnamed: 8', 'Unnamed: 9'],
      dtype='object')

(1000, 10)

## cleaning & normalized

In [14]:
#cleaning

#drop kolom ga guna
drop_cols = ["Unnamed: 8", "Unnamed: 9", "label_suggest", "label_suggest_v2"]
df_raw = df_raw.drop(columns=[c for c in drop_cols if c in df_raw.columns])

# format row_id biar sama kek dapin jdi DETIK_XXXX
df_raw["row_id"] = (
    df_raw["row_id"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
)
df_raw["row_id"] = pd.to_numeric(df_raw["row_id"], errors="coerce").astype("Int64")

df_raw["row_id"] = df_raw["row_id"].apply(lambda x: f"DETIK_{int(x):04d}" if pd.notna(x) else pd.NA)




In [15]:
#bersihin kata kata iklan di content
# (case-insensitive, hapus variasi spasi)
PATTERNS = [
    r"\bsimak\s+video\b",
    r"\bscroll\s+to\s+continue\s+with\s+content\b"
]

def remove_phrases(text: str) -> str:
    if pd.isna(text):
        return ""
    s = str(text)
    for p in PATTERNS:
        s = re.sub(p, " ", s, flags=re.IGNORECASE)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df_raw["content"] = df_raw["content"].apply(remove_phrases)

# timpa kolom text pakai content yg dh cleannn (kalo nimpa dipikir mas)
df_raw["text"] = (df_raw["title"].astype(str).str.strip() + ". " + df_raw["content"].astype(str).str.strip()).str.strip()


In [16]:
# cek hasil
display(df_raw.columns)
display(df_raw.head(3))

Index(['date', 'title', 'content', 'text', 'label', 'row_id'], dtype='object')

,date,title,content,text,label,row_id
0,2023-12-20,mendagri copot pj bupati kampar karena tak net...,menteri dalam negeri mendagri tito karnavian m...,mendagri copot pj bupati kampar karena tak net...,negatif,DETIK_7302
1,2025-06-20,waka mpr nilai keputusan prabowo absen di g7 j...,"wakil ketua mpr dari fraksi pan, eddy soeparno...",waka mpr nilai keputusan prabowo absen di g7 j...,netral,DETIK_1543
2,2024-02-14,"setelah nyoblos, ahy bakal merapat ke istora s...",tim kampanye nasional tkn prabowo-gibran bakal...,"setelah nyoblos, ahy bakal merapat ke istora s...",netral,DETIK_6055


In [17]:
#normalized 

df_raw["label"] = df_raw["label"].astype(str).str.lower().str.strip()

label_map = {
    "positif": 1,
    "negatif": -1,
    "netral": 0
}

df_raw["label"] = df_raw["label"].map(label_map)

#rename row_id -> article_id
df_raw = df_raw.rename(columns={"row_id": "article_id"})


print(df_raw["label"].value_counts(dropna=False))
display(df_raw.head(3))


label
 1    363
-1    332
 0    305
Name: count, dtype: int64


,date,title,content,text,label,article_id
0,2023-12-20,mendagri copot pj bupati kampar karena tak net...,menteri dalam negeri mendagri tito karnavian m...,mendagri copot pj bupati kampar karena tak net...,-1,DETIK_7302
1,2025-06-20,waka mpr nilai keputusan prabowo absen di g7 j...,"wakil ketua mpr dari fraksi pan, eddy soeparno...",waka mpr nilai keputusan prabowo absen di g7 j...,0,DETIK_1543
2,2024-02-14,"setelah nyoblos, ahy bakal merapat ke istora s...",tim kampanye nasional tkn prabowo-gibran bakal...,"setelah nyoblos, ahy bakal merapat ke istora s...",0,DETIK_6055


In [18]:
df_raw.to_csv(
    "detik_cleaned_labeled_manual.csv",
    index=False,
    encoding="utf-8"
)